# Unanticipated timing shift (review item 4.2) — Colab run

**Question.** How much does C3 lose when a shift the screen did not anticipate also
moves features that C3 keeps? Does screening against that shift restore performance?

- **Base transformation:** the locked main operator (hybrid-768), unchanged.
- **Added:** an assumed handshake delay of 0, 0.1, 1, 10 or 100 ms on the timing
  features of eligible (bidirectional) flows. These delays are assumptions, not
  measurements.
- **Conditions:**
  - C1: original data;
  - C2(d): the full-feature model on shifted test data;
  - C3u(d): the paper's C3, whose screen did not see the delay;
  - C3a(d): the screen sees the delay, then features are re-selected and the
    model is refitted.
- **Fits:** 120. The run takes about 30–45 minutes. Zero delay must reproduce
  the locked main scores.
- **Fixed plan:** `environment/timing_shift_plan.json` (frozen before any fit)
  and `environment/timing_shift_plan_amendment_1.md`. The notebook refuses a
  package whose runner, analysis or plan differs.

This repeats the run already made in the session workspace. The last cell
compares the two runs score by score.

## Steps

1. Select **Runtime → Run all**.
2. Upload the **PQC_IDS_Reproduction_Working_Copy.zip** supplied with this
   notebook.
3. Connect the Drive that holds the four original CSV files
   (`/content/drive/MyDrive/cicids2018`). Change `DATA_DIR` only if they moved.
4. When the run finishes, the second cell prints the results and the comparison.
5. Return **PQC_IDS_Timing_Shift_Results.zip**, even if it records a failure. A
   copy is saved to `MyDrive/pqc_ids_timing_shift/<run_id>/`.

Prepared with assistant assistance. Locked results are not modified.


In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import stat
import subprocess
import sys
import tempfile
import traceback
import uuid
import zipfile
from google.colab import files, drive

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
SESSION = Path(tempfile.mkdtemp(prefix="pqc_timing_shift_", dir="/content"))

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


# Edit this folder only if the four original CSV files moved.
DATA_DIR = Path("/content/drive/MyDrive/cicids2018")
DRIVE_OUTPUT = SESSION / "records"
DRIVE_OUTPUT.mkdir()
RUN_OUTPUT = DRIVE_OUTPUT / "results"
SUMMARY = {"run_id": RUN_ID, "status": "preparing", "fits_completed": 0,
           "scope": "Unanticipated timing shift (review item 4.2), preserving historical records",
           "historical_results_modified": False, "colab_kernel_python": platform.python_version()}
DRIVE_CONNECTED = False

def save_summary():
    (DRIVE_OUTPUT / "notebook_status.json").write_text(json.dumps(SUMMARY, indent=2) + "\n")

save_summary()
try:
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("Upload exactly one updated working-copy ZIP.")
    archive_path = SESSION / "working_copy.zip"
    archive_path.write_bytes(uploaded[zip_names[0]])
    del uploaded
    extracted = SESSION / "package"
    with zipfile.ZipFile(archive_path) as archive:
        if archive.testzip() is not None:
            raise ValueError("The ZIP failed its integrity check. Download the working copy again.")
        for info in archive.infolist():
            path = PurePosixPath(info.filename)
            if path.is_absolute() or ".." in path.parts or stat.S_ISLNK(info.external_attr >> 16):
                raise ValueError("Unexpected archive path: " + info.filename)
        archive.extractall(extracted)
    manifest_paths = list(extracted.glob("*/package_manifest.json"))
    if len(manifest_paths) != 1:
        raise ValueError("Expected one working-copy package manifest.")
    PACKAGE = manifest_paths[0].parent
    manifest = json.loads(manifest_paths[0].read_text())
    listed = set()
    for item in manifest["files"]:
        path = (PACKAGE / item["path"]).resolve()
        if not path.is_relative_to(PACKAGE.resolve()):
            raise ValueError("Invalid manifest path.")
        if not path.is_file() or path.stat().st_size != item["bytes"] or sha256(path) != item["sha256"]:
            raise ValueError("Package file does not match the manifest: " + item["path"])
        listed.add(item["path"])
    actual = {p.relative_to(PACKAGE).as_posix() for p in PACKAGE.rglob("*") if p.is_file()}
    if actual != listed | {"package_manifest.json"} or len(listed) != len(manifest["files"]):
        raise ValueError("The archive contains unlisted files or duplicate manifest records.")
    if not (PACKAGE / "scripts/run_timing_shift.py").is_file():
        raise ValueError("This is an older working copy. Use the ZIP provided with this notebook.")
    # The timing-shift runner, analysis and fixed plan must match exactly.
    EXPECTED = {
        "scripts/run_timing_shift.py": "8cdb7656d49f66b37eafbb8388640ba36bc4bf87f79acd0ca2968e1a0da15be0",
        "scripts/analyse_timing_shift.py": "3547cc706360f8bdf96989ec26f6896c7caaf50408a6f05ac432aa9c7e40a947",
        "environment/timing_shift_plan.json": "acd132a1952069939f6f387bcbbb689967159d9c75d73d738e5db2563b507f11",
        "environment/timing_shift_plan_amendment_1.md": "0cf84d66f7797395e4fb06578e525a7b1e9d5084c935f87c479d6c0f4d531abf",
    }
    for rel, expected in EXPECTED.items():
        if sha256(PACKAGE / rel) != expected:
            raise ValueError("This working copy has a different " + rel + ". Use the ZIP supplied with this notebook.")
    RUNNER_SHA256 = EXPECTED["scripts/run_timing_shift.py"]
    print("Timing-shift runner, analysis and plan verified.")
    print("Working-copy files verified:", len(listed))
    print("Run identifier:", RUN_ID)

    SUMMARY["package_sha256"] = sha256(archive_path)
    SUMMARY["runner_sha256"] = RUNNER_SHA256
    SUMMARY["package_manifest_sha256"] = sha256(PACKAGE / "package_manifest.json")
    shutil.copy2(PACKAGE / "package_manifest.json", DRIVE_OUTPUT / "working_copy_manifest.json")
    for name in ("main_known_versions.json", "main_requirements.txt", "timing_shift_plan.json", "timing_shift_plan_amendment_1.md"):
        shutil.copy2(PACKAGE / "environment" / name, DRIVE_OUTPUT / name)
    TARGET = json.loads((PACKAGE / "environment/main_known_versions.json").read_text())
    drive.mount("/content/drive")
    DRIVE_CONNECTED = True
    SUMMARY["data_directory"] = str(DATA_DIR)
    SUMMARY["target"] = TARGET
    save_summary()

    RUN_ENV = os.environ.copy()
    for name in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV"):
        RUN_ENV.pop(name, None)
    RUN_ENV["PYTHONNOUSERSITE"] = "1"
    RUN_ENV["UV_PYTHON_INSTALL_DIR"] = str(SESSION / "python_runtimes")
    RUN_ENV["UV_CACHE_DIR"] = str(SESSION / "uv_cache")

    def run_logged(command, log_name, env):
        with (DRIVE_OUTPUT / log_name).open("w", encoding="utf-8") as log:
            log.write("Command: " + json.dumps([str(s) for s in command]) + "\n")
            log.flush()
            with subprocess.Popen([str(s) for s in command], stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                                  text=True, bufsize=1, env=env) as process:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                return process.wait()

    ENVIRONMENT_READY = False
    VENV = SESSION / "timing_shift_environment"
    FIT_PYTHON = VENV / "bin/python"
    try:
        tooling = SESSION / "uv_tooling"
        bootstrap = [sys.executable, "-m", "pip", "install", "--target", tooling,
                     "--no-deps", "uv==0.12.15"]
        if run_logged(bootstrap, "01_uv_install.log", RUN_ENV) != 0:
            raise RuntimeError("Could not install the pinned uv tool.")
        uv_env = RUN_ENV.copy()
        uv_env["PYTHONPATH"] = str(tooling)
        uv = [sys.executable, "-m", "uv"]
        create = uv + ["venv", "--python", TARGET["python"], "--managed-python", VENV]
        if run_logged(create, "02_python_setup.log", uv_env) != 0:
            raise RuntimeError("Could not create the requested Python runtime.")
        install = uv + ["pip", "install", "--python", FIT_PYTHON, "--only-binary", ":all:",
                        "-r", PACKAGE / "environment/main_requirements.txt"]
        if run_logged(install, "03_packages_install.log", uv_env) != 0:
            raise RuntimeError("Could not install the declared package versions.")
        ENVIRONMENT_READY = True
        SUMMARY["status"] = "environment_installed"
    except Exception as error:
        SUMMARY["status"] = "environment_setup_failed"
        SUMMARY["error"] = str(error)
        (DRIVE_OUTPUT / "setup_traceback.txt").write_text(traceback.format_exc())
        print("Setup failed:", error)
    finally:
        save_summary()
    print("Environment ready:", ENVIRONMENT_READY)

    if not ENVIRONMENT_READY:
        raise RuntimeError("The timing-shift run did not start because environment setup failed; return the exported logs.")
    SUMMARY["status"] = "running_timing_shift"
    save_summary()
    RUN_ENV["OPENBLAS_CORETYPE"] = "Haswell"  # same BLAS kernels as the locked runs
    returncode = run_logged([FIT_PYTHON, "-u", PACKAGE / "scripts/run_timing_shift.py",
                             "--data-dir", DATA_DIR, "--cache-dir", SESSION / "input_cache",
                             "--output", RUN_OUTPUT], "04_timing_shift.log", RUN_ENV)
    SUMMARY["runner_returncode"] = returncode
    status_path = RUN_OUTPUT / "status.json"
    if not status_path.is_file():
        raise RuntimeError("The runner produced no status record; return the ZIP.")
    run_status = json.loads(status_path.read_text())
    SUMMARY["runner_status"] = run_status["status"]
    SUMMARY["fits_completed"] = run_status.get("fits", 0)
    if returncode != 0 or run_status["status"] != "completed" or run_status.get("completed_seeds") != list(range(10)):
        raise RuntimeError("The timing-shift run did not complete; return the ZIP containing its failure evidence.")
    SUMMARY["status"] = "running_analysis"
    save_summary()
    code = run_logged([FIT_PYTHON, "-u", PACKAGE / "scripts/analyse_timing_shift.py", RUN_OUTPUT,
                       RUN_OUTPUT / "analysis"], "05_analysis.log", RUN_ENV)
    if code != 0:
        raise RuntimeError("The analysis step failed; return the ZIP.")
    SUMMARY["status"] = "timing_shift_completed"
    print("Completed fits:", run_status["fits"], "(expected 120)")
    print("Return the ZIP for independent checking; no historical scores were replaced.")

except BaseException as error:
    SUMMARY["status"] = "timing_shift_incomplete"
    SUMMARY["error"] = str(error)
    (DRIVE_OUTPUT / "notebook_traceback.txt").write_text(traceback.format_exc())
    print("Please return the ZIP with this recorded issue:", error)
finally:
    SUMMARY["exported_utc"] = datetime.now(timezone.utc).isoformat()
    save_summary()
    result_files = sorted(p for p in DRIVE_OUTPUT.rglob("*") if p.is_file())
    items = [{"path": p.relative_to(DRIVE_OUTPUT).as_posix(), "bytes": p.stat().st_size,
              "sha256": sha256(p)} for p in result_files]
    manifest_path = DRIVE_OUTPUT / "result_manifest.json"
    manifest_path.write_text(json.dumps({"run_id": RUN_ID, "files": items}, indent=2) + "\n")
    result_files.append(manifest_path)
    result_zip = SESSION / "PQC_IDS_Timing_Shift_Results.zip"
    with zipfile.ZipFile(result_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in result_files:
            archive.write(path, path.relative_to(DRIVE_OUTPUT).as_posix())
    with zipfile.ZipFile(result_zip) as archive:
        if archive.testzip() is not None:
            raise RuntimeError("Result archive failed its integrity check.")
    runner_status = RUN_OUTPUT / "status.json"
    if runner_status.is_file():
        captured = json.loads(runner_status.read_text())
        print("Runner recorded fits:", captured.get("fits", 0))
    if DRIVE_CONNECTED:
        try:
            saved = Path("/content/drive/MyDrive/pqc_ids_timing_shift") / RUN_ID
            saved.mkdir(parents=True, exist_ok=False)
            shutil.copy2(result_zip, saved / result_zip.name)
            print("Drive copy:", saved / result_zip.name)
        except OSError as error:
            print("Drive copy failed, but the downloadable ZIP is available:", error)
    print("Final status:", SUMMARY["status"])
    print("Fits recorded:", SUMMARY.get("fits_completed", "see runner status"))
    print("ZIP SHA-256:", sha256(result_zip))
    files.download(str(result_zip))


In [ ]:
# Results: printed from this run's analysis files, plus a score-by-score comparison
# with the run made in the session workspace (verified_outputs/timing_shift/metrics.csv).
import pandas as pd
pd.set_option("display.width", 200)
A = RUN_OUTPUT / "analysis"
print(json.dumps(json.loads((A / "replay.json").read_text()), indent=2))
zero = json.loads((RUN_OUTPUT / "zero_delay_vs_locked.json").read_text())
print("Largest zero-delay difference from the locked record:",
      max(max(r["C1"], r["C2"], r["C3"]) for r in zero))

s = pd.read_csv(A / "timing_shift_statistics.csv")
cols = ["delay_ms", "model", "n_features", "mean_difference", "ci90_lower", "ci90_upper",
        "tost_0.01_holm", "equiv_0.01", "equiv_0.005", "corr_equiv_0.01"]
for cond, var, title in [("C3u", "additive_max", "C3 - C1, screen did NOT see the delay (primary)"),
                         ("C3u", "floor_max", "C3 - C1, screen did NOT see the delay (floor variant)"),
                         ("C3a", "additive_max", "C3 - C1, screen DID see the delay"),
                         ("C2", "additive_max", "C2 - C1, full-feature model")]:
    print("\n" + title)
    t = s[(s.condition == cond) & (s.variant == var)]
    print(t[[c for c in cols if c in t]].round(4).to_string(index=False))
print("\nTolerance summary:")
print(json.dumps(json.loads((A / "tolerance_summary.json").read_text()), indent=1))

# Compare with the workspace run
colab = pd.read_csv(RUN_OUTPUT / "metrics.csv")
work = pd.read_csv(PACKAGE / "verified_outputs/timing_shift/metrics.csv")
key = ["seed", "model", "condition", "variant", "delay_us"]
both = colab.merge(work, on=key, suffixes=("_colab", "_workspace"))
both["diff"] = (both.weighted_f1_colab - both.weighted_f1_workspace).abs()
print("\nRows compared:", len(both), "of", len(colab))
print("Identical scores:", int((both["diff"] == 0).sum()))
print(both.groupby("model")["diff"].max().rename("largest |difference|"))
print(both[both["diff"] > 0][key + ["weighted_f1_colab", "weighted_f1_workspace"]].head(20).to_string(index=False))
